<a href="https://colab.research.google.com/github/hudamohmand/Project_3_Demand_Estimation/blob/main/part2_air_fryer_data_scientist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Demand Estimation — Air Fryers


We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.



In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [8]:
df = pd.read_csv("air_fryers_clean_brand_year.csv")

print(df.shape)
print(df.head())
print(df.groupby("year")["brand_share"].sum())

feature_cols = [
    "compact_share",
    "dual_basket_share",
    "oven_style_share",
    "rotisserie_share",
    "window_share",
]

(50, 15)
     category  year       brand  purchase_count  product_count   avg_price  \
0  air_fryers  2019     chefman            1146             10   72.963695   
1  air_fryers  2019      cosori              11              2  159.990000   
2  air_fryers  2019   cuisinart            1616             22  229.465274   
3  air_fryers  2019        dash            3011             19   55.176333   
4  air_fryers  2019  gowise usa            4405             45   83.575551   

   avg_rating  compact_share  dual_basket_share  oven_style_share  \
0    4.434119       1.000000                0.0          0.780977   
1    4.581818       1.000000                0.0          0.090909   
2    4.481312       0.993812                0.0          0.889851   
3    4.390767       1.000000                0.0          0.973431   
4    4.552259       0.999773                0.0          0.129398   

   rotisserie_share  window_share  market_purchases  brand_share  \
0          0.243455      0.184119      

In [9]:
y = df["log_brand_share"]

brand_dummies = pd.get_dummies(df["brand"],
                               prefix="brand", drop_first=True, dtype=int)

year_dummies = pd.get_dummies(df["year"].astype(str),
                              prefix="year", drop_first=True, dtype=int)

X = pd.concat(
    [df[["avg_price", "avg_rating"] + feature_cols],
     brand_dummies,
     year_dummies],
    axis=1,
)

model = LinearRegression()
model.fit(X, y)

predicted_log_share = model.predict(X)
r2 = r2_score(y, predicted_log_share)

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_
})

print("R-squared:", r2)
coef_table

R-squared: 0.763453950091436


,feature,coefficient
0,avg_price,-0.037668
1,avg_rating,0.287517
2,compact_share,9.815304
3,dual_basket_share,-9.509686
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
6,window_share,12.880298
7,brand_cosori,2.551946
8,brand_cuisinart,6.422436
9,brand_dash,0.176655


In [14]:
price_coef = coef_table.loc[coef_table["feature"] == "avg_price", "coefficient"].iloc[0]
rating_coef = coef_table.loc[coef_table["feature"] == "avg_rating", "coefficient"].iloc[0]

feature_coeffs = coef_table[coef_table["feature"].isin(feature_cols)].sort_values(
    "coefficient", ascending=False
)

brand_coeffs = coef_table[coef_table["feature"].str.startswith("brand_")].sort_values(
    "coefficient", ascending=False
)

year_coeffs = coef_table[coef_table["feature"].str.startswith("year_")].sort_values(
    "coefficient", ascending=False
)

print("Price coefficient:", price_coef)
print("Average rating coefficient:", rating_coef)
print("R-squared:", r2)
print("\nProduct feature coefficients:")
display(feature_coeffs)

print("\nBrand dummy coefficients:")
display(brand_coeffs)

print("\nYear dummy coefficients:")
display(year_coeffs)


Price coefficient: -0.03766765298429383
Average rating coefficient: 0.287516847432701
R-squared: 0.763453950091436

Product feature coefficients:


,feature,coefficient
6,window_share,12.880298
2,compact_share,9.815304
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
3,dual_basket_share,-9.509686



Brand dummy coefficients:


,feature,coefficient
8,brand_cuisinart,6.422436
12,brand_ninja,5.838705
11,brand_instant_pot,4.626260
10,brand_gowise usa,3.938996
14,brand_oster,3.928074
13,brand_nuwave,3.544883
7,brand_cosori,2.551946
15,brand_ultrean,0.942399
9,brand_dash,0.176655



Year dummy coefficients:


,feature,coefficient
16,year_2020,0.119071
17,year_2021,0.041900
19,year_2023,-0.003307
18,year_2022,-0.098860


In [17]:
print("Dropped/reference brand:", sorted(df["brand"].unique())[0])
print("Dropped/reference year:", sorted(df["year"].unique())[0])

Dropped/reference brand: chefman
Dropped/reference year: 2019


Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
**Answer:** The estimated price coefficient is about -0.0377. It was estimated using the fitted linear regression model, after creating a predictor matrix with the features, the coefficient was extracted with the line of code 'avg_price' from the model's coefficient table.


2. Is it negative? Why is that important? **Answer:** The estimated price coefficient is negative and it matters because it implies that as price for this product increases, demand should decrease. This means that holding ratings, product features, brand effects, and year effects are constant and higher prices are associated with lower log brand share. This matters when determining the demand of a product and pricing of a product.


3. Which product features are associated with higher demand? **Answer:** The product features associated with higher demand are window_share, compact_share, and oven_style_share, because these have positive coefficients. The features associated with lower demand are rotisserie_share and dual_basket_share, because these have negative coefficients.


4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand. **Answer:** The largest brand dummy coefficients are brand_cuisinart, brand_ninja, and brand_instant_pot relative to the dropped reference brand, chefman. Meaning these brands have higher demand than Chefman after controlling for price, rating, features, and year effects.


5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year. **Answer:** The largest year dummy coefficient is year_2020 and year_2021. These are interpreted relative to the dropped reference year, 2019. This means demand was estimated to be higher in 2020 and 2021 than in 2019.


6. What is the model's $R^2$? **Answer:** The model's $R^2$ is approximately 0.7635, meaning the model explains about 76.3% of the variation in log brand market share.

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.